# Clogging trajectories: data visualization

Plots the raw Monte-Carlo clogging trajectories (`{SG}_MC_UNIFORM_HL/CL.npy`) against field
data (`{SG}_FIELD_DATA.csv`) for each steam generator in `DATA/`, using the chemistry
conditioning/cleaning metadata (`{SG}_METADATA.json`) to annotate the plot — adapted from the
`plot_tpd` function in `bayes-calibration-for-prognostics/plot.py`.

Only depends on `data_preparation.py` (already in this repo) plus numpy/matplotlib — no
OpenTURNS needed, so trajectories can be inspected without running the (expensive)
assimilation pipeline.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np

from clogging_prognostics.data_preparation import prepro_data_kl, preprocess_field_data

BASE_DIRECTORY = 'DATA/'
INPUT_DIMENSION = 7   # alpha, beta, eps_e, eps_c, d_p, g_p0, a_v (+ time column)
SG_TAGS = ['SGA', 'SGB']
COLORS = {'SGA': 'tab:red', 'SGB': 'tab:blue'}
SG_DISPLAY = {'SGA': 'SG A', 'SGB': 'SG B'}

## Load the raw data for one steam generator

In [ ]:
def load_sg_data(sg_tag, base_directory=BASE_DIRECTORY):
    """Load MC trajectories, metadata and field data for one steam generator."""
    p = lambda name: os.path.join(base_directory, f'{sg_tag}_{name}')

    simulation_time = np.load(p('TIME.npy'), allow_pickle=True)
    input_dic = json.load(open(p('METADATA.json')))

    mc_hl = np.load(p('MC_UNIFORM_HL.npy'), allow_pickle=True)
    mc_cl = np.load(p('MC_UNIFORM_CL.npy'), allow_pickle=True)
    n_max = mc_hl.shape[0]

    # X, y: DoE input samples / clogging trajectories (unscaled -> only for plotting)
    X, y = prepro_data_kl(mc_hl, mc_cl, n_max, n_max, INPUT_DIMENSION, scale=False)

    cleaning = input_dic['chemistry']['cleaning']
    cleaning_dates = [c['time'] / 24 for c in cleaning]
    field_data, field_time_indices = preprocess_field_data(
        csv_file=p('FIELD_DATA.csv'),
        operation_time=simulation_time,
        cleaning_dates=cleaning_dates,
        with_time_division=False,
    )
    # field_time_indices = [esticol_row_positions, etv_row_positions] into field_data
    field_esticol = field_data[field_time_indices[0]]
    field_etv = field_data[field_time_indices[1]]

    return {
        'simulation_time': simulation_time,
        'input_dic': input_dic,
        'X': X,
        'y': y,
        'field_esticol': field_esticol,
        'field_etv': field_etv,
    }

datasets = {sg_tag: load_sg_data(sg_tag) for sg_tag in SG_TAGS}
for sg_tag, data in datasets.items():
    print(f"{sg_tag}: y {data['y'].shape}, simulation_time {data['simulation_time'].shape}, "
          f"ESTICOL points {data['field_esticol'].shape[0]}, ETV points {data['field_etv'].shape[0]}")

## `plot_tpd`

Adapted from `bayes-calibration-for-prognostics/plot.py`: shades the chemistry-conditioning
windows, draws the clogging-trajectory bundle, marks curative/preventive cleanings, and
(new) overlays the field-data points, colored by measurement type -- ESTICOL vs. ETV.

In [ ]:
def plot_tpd(axs, input_dic, y, simulation_time, label, color,
             field_esticol=None, field_etv=None):
    """Plot the clogging trajectories against chemistry/cleaning metadata.

    Parameters
    ----------
    input_dic: dict
        metadata dictionary (chemistry conditioning/cleaning), as in `{SG}_METADATA.json`
    y: numpy.ndarray
        clogging trajectories, shape (n_samples, n_time)
    simulation_time: numpy.ndarray
        simulation time grid, in days
    label: str
        legend label for the trajectory bundle
    color: str
        matplotlib color for the trajectory bundle
    field_esticol, field_etv: numpy.ndarray, optional
        (n_obs, 2) arrays of (time, tau_c) field observations, plotted in distinct
        colors/markers so the two measurement types are visually separable

    Returns
    -------
    axs: matplotlib.axes.Axes
    """
    colors_chem_cond = {'C1': {'name': r'$\chi_{1}$', 'low': 'gainsboro', 'high': 'silver'},
                      'C2': {'name': r'$\chi_{2}$', 'high': 'darkgray'}}

    # Chemical conditioning
    cond = input_dic['chemistry']['conditioning']
    for k in range(len(cond)):
        if k == len(cond)-1:
            t_i, t_f = cond[k]['time']/24, max(simulation_time)
        else:
            t_i, t_f = cond[k]['time']/24, cond[k+1]['time']/24
        c = cond[k]
        typ = colors_chem_cond[c['type']]
        col = typ[str(c['ph'])]
        name = typ['name']
        axs.axvspan(t_i, t_f, facecolor=col, alpha=0.8, label='{0}, {1} pH'.format(name, c['ph']))

    # Clogging trajectories
    axs.grid()
    n = y.shape[0]
    d = y.shape[1]

    if d > 1:
        for i in range(n):
            axs.plot(simulation_time, y[i], c=color, alpha=0.1)
            if i == n-1:
                axs.plot(simulation_time, y[i], c=color, alpha=0.5, label=label)
    else:
        axs.scatter(simulation_time, y, c='g', s=5, label=label, zorder=10)

    # Field data overlay, colored by measurement type. Picked to stay legible
    # against both the red/blue trajectory bundles and the gray chemistry shading.
    if field_esticol is not None and len(field_esticol):
        axs.scatter(field_esticol[:, 0], field_esticol[:, 1], c='lime', marker='o',
                    edgecolors='k', linewidths=0.4, s=16, zorder=11, label='ESTICOL')
    if field_etv is not None and len(field_etv):
        axs.scatter(field_etv[:, 0], field_etv[:, 1], c='magenta', marker='^',
                    edgecolors='k', linewidths=0.4, s=22, zorder=12, label='ETV')

    axs.set_xlim(min(simulation_time), max(simulation_time))
    axs.set_ylim(0, 100)

    axs.set_xlabel(r'$t\;(d)$', fontsize=15)
    axs.set_ylabel(r'$\tau_c$', fontsize=15)

    # Cleaning dates
    nett = input_dic['chemistry']['cleaning']
    for k in range(len(nett)):
        if nett[k]['type'] == 'curative':
            axs.axvline(x=nett[k]['time']/24, color='k', linestyle='-', linewidth=1.5, label='Curative cleaning')
        elif nett[k]['type'] == 'preventive':
            axs.axvline(x=nett[k]['time']/24, color='k', linestyle='--', linewidth=1.5, label='Preventive cleaning')

    return axs

## One steam generator at a time

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(10, 6), layout='tight')
sg_tag = 'SGA'
data = datasets[sg_tag]
plot_tpd(axs, data['input_dic'], data['y'], data['simulation_time'],
         label=f'MC trajectories ({SG_DISPLAY[sg_tag]})', color=COLORS[sg_tag],
         field_esticol=data['field_esticol'], field_etv=data['field_etv'])

handles, labels = axs.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axs.legend(by_label.values(), by_label.keys(), fontsize=9, loc='upper left')
axs.set_title(SG_DISPLAY[sg_tag])
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(10, 6), layout='tight')
sg_tag = 'SGB'
data = datasets[sg_tag]
plot_tpd(axs, data['input_dic'], data['y'], data['simulation_time'],
         label=f'MC trajectories ({SG_DISPLAY[sg_tag]})', color=COLORS[sg_tag],
         field_esticol=data['field_esticol'], field_etv=data['field_etv'])

handles, labels = axs.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axs.legend(by_label.values(), by_label.keys(), fontsize=9, loc='upper left')
axs.set_title(SG_DISPLAY[sg_tag])
plt.show()

## Side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), layout='tight', sharey=True)

for ax, sg_tag in zip(axes, SG_TAGS):
    data = datasets[sg_tag]
    plot_tpd(ax, data['input_dic'], data['y'], data['simulation_time'],
             label=f'MC trajectories ({SG_DISPLAY[sg_tag]})', color=COLORS[sg_tag],
             field_esticol=data['field_esticol'], field_etv=data['field_etv'])
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), fontsize=8, loc='upper left')
    ax.set_title(SG_DISPLAY[sg_tag])

plt.show()

## RUL-vs-observation-size figures (already computed, in `DATA/`)

the two SG B figures (`rul_vs_obs_size_SGB.png` and `rul_cv_vs_obs_size_SGB.png`) were produced by the
`run_observation_size` sweep (see `rul_observation_sweep.ipynb`); displayed here for reference
rather than recomputed.

In [ ]:
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fname, title in zip(axes, ['rul_vs_obs_size_SGB.png', 'rul_cv_vs_obs_size_SGB.png'],
                            ['SG B: RUL vs. observation size', 'SG B: RUL CV vs. observation size']):
    path = os.path.join(BASE_DIRECTORY, fname)
    ax.imshow(Image.open(path))
    ax.axis('off')
    ax.set_title(title)
plt.tight_layout()
plt.show()